# 05 - Risk-Neutral Drift Enforcement

Goal: recompute drift so probability $p_t = S(x_t)$ is martingale-consistent.

Using $S(x)=1/(1+e^{-x})$, a practical drift correction is:
$$
\mu_t \approx \frac{1}{2}(1-2p_t)\sigma_{b,t}^2 + \frac{\mathbb E[S(x_t+Z)-S(x_t)-S'(x_t)Z]}{S'(x_t)}
$$
where $Z$ is jump size under the estimated jump law.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [3]:
inp = Path('stage4_em.csv')
if not inp.exists():
    raise FileNotFoundError('Run notebook 04 first to generate stage4_em.csv')

df = pd.read_csv(inp)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
df = df.sort_values('timestamp').reset_index(drop=True)
df.head()

,timestamp,p_clipped,y_logit,x_hat,dx,jump_prob,is_jump_dominant,sigma2_b,sigma2_j,lambda_jump
0,2025-11-14 16:31:16+00:00,0.977495,3.771256,3.771256,0.0,0.05,False,0.0,0.0,0.05


## Compute RN drift term

In [4]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

x = df['x_hat'].to_numpy(float)
p = sigmoid(x)
sig2 = np.maximum(df['sigma2_b'].to_numpy(float), 1e-10)
lam = np.maximum(df['lambda_jump'].to_numpy(float), 1e-12)
sj2 = np.maximum(df['sigma2_j'].to_numpy(float), 1e-10)

# Diffusion component from Ito expansion
mu_diff = 0.5 * (1.0 - 2.0 * p) * sig2

# Jump correction via Gaussian jump law approximation
rng = np.random.default_rng(7)
z = rng.normal(0.0, np.sqrt(sj2), size=len(df))
s1 = p
s1p = s1 * (1.0 - s1)
jump_num = sigmoid(x + z) - s1 - s1p * z
mu_jump = lam * (jump_num / np.maximum(s1p, 1e-8))

df['mu_rn'] = mu_diff + mu_jump
df[['mu_rn']].describe().T

,count,mean,std,min,25%,50%,75%,max
mu_rn,1.0,-4.774939e-11,NaN,-4.774939e-11,-4.774939e-11,-4.774939e-11,-4.774939e-11,-4.774939e-11


## Re-smooth latent path with RN drift in transition

Transition used here: $x_t = x_{t-1} + u_{t-1} + ta_t$.

In [5]:
y = df['y_logit'].to_numpy(float)
r = np.maximum((df['sigma2_b'] * 0.25).to_numpy(float), 1e-8)
mu = df['mu_rn'].to_numpy(float)

x_rn = np.zeros(len(df))
p_var = np.zeros(len(df))
x_prev = y[0]
p_prev = np.var(y[: min(100, len(y))]) + 1e-4

for t in range(len(df)):
    q_t = np.maximum(df['sigma2_b'].iloc[t], 1e-8)
    x_pred = x_prev + (mu[t - 1] if t > 0 else 0.0)
    p_pred = p_prev + q_t

    k = p_pred / (p_pred + r[t])
    x_new = x_pred + k * (y[t] - x_pred)
    p_new = (1 - k) * p_pred

    x_rn[t] = x_new
    p_var[t] = p_new
    x_prev, p_prev = x_new, p_new

df['x_hat_rn'] = x_rn
df['kalman_var_rn'] = p_var
df[['x_hat', 'x_hat_rn']].head()

,x_hat,x_hat_rn
0,3.771256,3.771256


In [6]:
out_cols = [
    'timestamp', 'p_clipped', 'x_hat', 'sigma2_b', 'sigma2_j', 'lambda_jump',
    'mu_rn', 'x_hat_rn', 'kalman_var_rn', 'jump_prob', 'is_jump_dominant',
]
df[out_cols].to_csv('stage5_rn.csv', index=False)
print('saved:', Path('stage5_rn.csv').resolve())
print('rows:', len(df))

saved: C:\Users\p\Documents\GitHub\volatility-estimator\research\stage5_rn.csv
rows: 1
